<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 28 - Pig y Hive (Práctica avanzada)

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Crear **UDFs** (funciones personalizadas) simulando Pig y Hive
- Implementar **particionado** y **bucketing** simulados
- Construir un **pipeline ETL completo** Pig→Hive
- Comparar formatos de almacenamiento

## 1. Preparación: datos raw con errores

Creamos un dataset "sucio" que simula datos reales que necesitan procesamiento ETL.

In [ ]:
import pandas as pd
import sqlite3
import random
import os
import json
import time

random.seed(42)

categorias = ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros']
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
productos_por_cat = {
    'Electrónica': ['Portátil', 'Móvil', 'Tablet', 'Auriculares'],
    'Ropa': ['Camiseta', 'Pantalón', 'Chaqueta', 'Zapatos'],
    'Hogar': ['Lámpara', 'Silla', 'Mesa', 'Alfombra'],
    'Deportes': ['Balón', 'Raqueta', 'Bicicleta', 'Mancuernas'],
    'Libros': ['Novela', 'Manual técnico', 'Cómic', 'Biografía']
}

n = 1000
datos_raw = []
for i in range(n):
    cat = random.choice(categorias)
    prod = random.choice(productos_por_cat[cat])
    importe = round(random.uniform(5.0, 500.0), 2)

    # Introducir errores deliberados (~10% datos sucios)
    if random.random() < 0.03:
        importe = None
    if random.random() < 0.03:
        importe = -abs(round(random.uniform(1, 100), 2))
    if random.random() < 0.02:
        cat = cat.upper()  # inconsistencia en mayúsculas
    if random.random() < 0.02:
        prod = ''  # producto vacío

    datos_raw.append({
        'id': i + 1,
        'producto': prod,
        'categoria': cat,
        'importe': importe,
        'ciudad': random.choice(ciudades),
        'cliente_id': random.randint(1, 200),
        'fecha': f'2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}'
    })

# Añadir duplicados
for _ in range(20):
    datos_raw.append(random.choice(datos_raw).copy())

random.shuffle(datos_raw)
raw_df = pd.DataFrame(datos_raw)

print("=== DATOS RAW (con errores) ===")
print(f"Total registros: {len(raw_df)}")
print(f"Valores nulos en importe: {raw_df['importe'].isna().sum()}")
print(f"Importes negativos: {(raw_df['importe'] < 0).sum()}")
print(f"Productos vacíos: {(raw_df['producto'] == '').sum()}")
print(f"Categorías únicas (con inconsistencias): {raw_df['categoria'].nunique()}")
print(f"Duplicados: {raw_df.duplicated(subset=['id']).sum()}")
print("\nMuestra:")
print(raw_df.head(10).to_string(index=False))

## 2. UDFs (User Defined Functions)

### Simulando UDFs de Pig con funciones Python + `.apply()`

En Pig registraríamos una UDF con `REGISTER`. Aquí definimos funciones Python y las aplicamos con Pandas.

In [ ]:
# === UDF 1: Clasificar importe en rangos ===
# Pig: REGISTER 'udfs.py' USING jython AS misudf;
#      FOREACH ventas GENERATE misudf.clasificar_importe(importe);

def udf_clasificar_importe(importe):
    """UDF: clasifica importe en categorías de gasto"""
    if pd.isna(importe) or importe < 0:
        return 'INVALIDO'
    elif importe < 25:
        return 'micro'
    elif importe < 100:
        return 'bajo'
    elif importe < 250:
        return 'medio'
    else:
        return 'alto'

# === UDF 2: Normalizar categoría ===
def udf_normalizar_categoria(cat):
    """UDF: normaliza el formato de la categoría"""
    if not cat or not isinstance(cat, str):
        return 'Desconocida'
    return cat.strip().title()

# === UDF 3: Extraer trimestre de fecha ===
def udf_trimestre(fecha):
    """UDF: extrae el trimestre de una fecha YYYY-MM-DD"""
    try:
        mes = int(fecha.split('-')[1])
        return f'Q{(mes - 1) // 3 + 1}'
    except (ValueError, IndexError, AttributeError):
        return 'Q?'

# Aplicar UDFs (como FOREACH ... GENERATE en Pig)
raw_df['rango_importe'] = raw_df['importe'].apply(udf_clasificar_importe)
raw_df['categoria_norm'] = raw_df['categoria'].apply(udf_normalizar_categoria)
raw_df['trimestre'] = raw_df['fecha'].apply(udf_trimestre)

print("=== RESULTADO DE APLICAR UDFs ===")
print(raw_df[['producto', 'categoria', 'categoria_norm', 'importe',
              'rango_importe', 'fecha', 'trimestre']].head(15).to_string(index=False))

print("\n--- Distribución por rango de importe ---")
print(raw_df['rango_importe'].value_counts().to_string())

print("\n--- Distribución por trimestre ---")
print(raw_df['trimestre'].value_counts().sort_index().to_string())

In [ ]:
# === Simulando UDFs en Hive con SQLite ===
conn = sqlite3.connect(':memory:')

def sqlite_clasificar(importe):
    if importe is None or importe < 0:
        return 'INVALIDO'
    elif importe < 25:
        return 'micro'
    elif importe < 100:
        return 'bajo'
    elif importe < 250:
        return 'medio'
    return 'alto'

def sqlite_trimestre(fecha):
    try:
        mes = int(fecha.split('-')[1])
        return f'Q{(mes - 1) // 3 + 1}'
    except:
        return 'Q?'

conn.create_function("clasificar_importe", 1, sqlite_clasificar)
conn.create_function("trimestre", 1, sqlite_trimestre)

raw_df.to_sql('ventas_raw', conn, index=False, if_exists='replace')

print("=== HIVEQL con UDFs personalizadas ===")
query_udf = """
SELECT
    clasificar_importe(importe) AS rango,
    trimestre(fecha) AS trim,
    COUNT(*) AS cantidad,
    ROUND(AVG(CASE WHEN importe > 0 THEN importe END), 2) AS media
FROM ventas_raw
GROUP BY clasificar_importe(importe), trimestre(fecha)
ORDER BY rango, trim;
"""
resultado_udf = pd.read_sql(query_udf, conn)
print(resultado_udf.to_string(index=False))

## 3. Pipeline ETL completo: Pig (limpieza) → Hive (análisis)

Simulamos un pipeline ETL real:
1. **EXTRACT** (Pig LOAD): cargar datos raw
2. **TRANSFORM** (Pig FILTER/FOREACH): limpiar, normalizar, enriquecer
3. **LOAD** (Hive INSERT): cargar en tabla particionada

In [ ]:
print("="*60)
print("FASE 1 — EXTRACT (Pig LOAD)")
print("="*60)
raw_df.to_csv('datos_crudos.csv', index=False)
datos = pd.read_csv('datos_crudos.csv')
print(f"Registros cargados: {len(datos)}")
print(f"Columnas: {list(datos.columns)}")

print(f"\n{'='*60}")
print("FASE 2 — TRANSFORM (Pig FILTER + FOREACH)")
print("="*60)

antes = len(datos)

# Paso 2a: Eliminar duplicados (DISTINCT)
datos = datos.drop_duplicates(subset=['id'])
print(f"  [DISTINCT] Duplicados eliminados: {antes - len(datos)}")

# Paso 2b: Filtrar registros inválidos (FILTER)
antes = len(datos)
datos = datos.dropna(subset=['importe'])
datos = datos[datos['importe'] > 0]
datos = datos[datos['producto'] != '']
print(f"  [FILTER] Registros inválidos eliminados: {antes - len(datos)}")

# Paso 2c: Normalizar campos (FOREACH ... GENERATE con UDFs)
datos['categoria'] = datos['categoria'].apply(udf_normalizar_categoria)
datos['rango'] = datos['importe'].apply(udf_clasificar_importe)
datos['trimestre'] = datos['fecha'].apply(udf_trimestre)
datos['año'] = datos['fecha'].apply(lambda f: int(f.split('-')[0]))
datos['mes'] = datos['fecha'].apply(lambda f: int(f.split('-')[1]))
print(f"  [FOREACH] Campos derivados añadidos: rango, trimestre, año, mes")

print(f"\n  Registros limpios: {len(datos)}")
print(datos.head(10).to_string(index=False))

print(f"\n{'='*60}")
print("FASE 3 — LOAD (Hive INSERT INTO partitioned table)")
print("="*60)

conn_etl = sqlite3.connect(':memory:')
datos.to_sql('ventas_limpias', conn_etl, index=False, if_exists='replace')
print(f"Tabla 'ventas_limpias' creada con {len(datos)} registros")

# Mostrar las "particiones" (simuladas)
particiones = datos.groupby(['año', 'mes']).size().reset_index(name='registros')
print(f"\nParticiones simuladas (año/mes):")
print(particiones.to_string(index=False))

In [ ]:
# === CONSULTAS ANALÍTICAS sobre la tabla limpia (HiveQL) ===

print("=== CONSULTA 1: Ventas por categoría y trimestre ===")
q1 = """
SELECT
    categoria,
    trimestre,
    COUNT(*) AS ventas,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS media
FROM ventas_limpias
GROUP BY categoria, trimestre
ORDER BY categoria, trimestre;
"""
print(pd.read_sql(q1, conn_etl).to_string(index=False))

print("\n=== CONSULTA 2: Top 5 productos por ingreso ===")
q2 = """
SELECT
    producto,
    categoria,
    COUNT(*) AS unidades,
    ROUND(SUM(importe), 2) AS ingreso_total
FROM ventas_limpias
GROUP BY producto, categoria
ORDER BY ingreso_total DESC
LIMIT 5;
"""
print(pd.read_sql(q2, conn_etl).to_string(index=False))

print("\n=== CONSULTA 3: Distribución de rangos por ciudad ===")
q3 = """
SELECT
    ciudad,
    rango,
    COUNT(*) AS cantidad,
    ROUND(SUM(importe), 2) AS total
FROM ventas_limpias
GROUP BY ciudad, rango
ORDER BY ciudad, rango;
"""
print(pd.read_sql(q3, conn_etl).to_string(index=False))

## 4. Simulación de particionado

Simulamos el efecto del particionado guardando datos en carpetas separadas y comparando el rendimiento de las consultas.

In [ ]:
# === Simular tabla particionada con directorios ===
os.makedirs('tabla_particionada', exist_ok=True)

for (año, mes), grupo in datos.groupby(['año', 'mes']):
    ruta = f'tabla_particionada/año={año}/mes={mes:02d}'
    os.makedirs(ruta, exist_ok=True)
    grupo.to_csv(f'{ruta}/datos.csv', index=False)

print("=== Estructura de tabla particionada ===")
for root, dirs, files in os.walk('tabla_particionada'):
    nivel = root.replace('tabla_particionada', '').count(os.sep)
    indent = '  ' * nivel
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  └── {f}")

# === Comparar: consulta sin partición vs con partición ===
print("\n=== Comparativa de rendimiento ===")

# Sin partición: leemos TODO
t0 = time.time()
all_data = pd.read_csv('datos_crudos.csv')
result_sin = all_data[(all_data['fecha'].str.startswith('2025-06'))]
t_sin = time.time() - t0

# Con partición: leemos SOLO la partición relevante
t0 = time.time()
ruta_particion = 'tabla_particionada/año=2025/mes=06'
if os.path.exists(f'{ruta_particion}/datos.csv'):
    result_con = pd.read_csv(f'{ruta_particion}/datos.csv')
else:
    result_con = pd.DataFrame()
t_con = time.time() - t0

print(f"Sin partición: leído {len(all_data)} registros, filtrado a {len(result_sin)} → {t_sin*1000:.2f} ms")
print(f"Con partición: leído directamente {len(result_con)} registros → {t_con*1000:.2f} ms")
if t_con > 0:
    print(f"Mejora teórica: la consulta particionada lee ~{len(result_con)}/{len(all_data)} = {len(result_con)/len(all_data)*100:.1f}% de los datos")

## 5. Ejercicios propuestos

### Ejercicio 1: UDF personalizada
Crea una UDF que clasifique las ciudades en regiones geográficas:
- Madrid → Centro
- Barcelona → Cataluña
- Valencia → Levante
- Sevilla → Andalucía
- Bilbao → País Vasco

Aplícala al dataset y analiza ventas por región.

### Ejercicio 2: ETL con datos nuevos
Crea un dataset de logs de servidor (timestamp, método, URL, código_respuesta, tiempo_ms) con errores, y construye un pipeline ETL completo.

### Ejercicio 3: Consulta con bucketing simulado
Divide los datos limpios en 4 "buckets" por `cliente_id % 4` y demuestra cómo un JOIN entre dos tablas con el mismo bucketing solo necesita leer 1/4 de los datos.

In [ ]:
# === EJERCICIO 1: UDF de regiones ===
# Tu código aquí:
# def udf_region(ciudad): ...
# datos['region'] = datos['ciudad'].apply(udf_region)


# === EJERCICIO 2: ETL de logs ===
# Tu código aquí:
# 1. Generar datos de logs con errores
# 2. Limpiar con Pandas (Pig)
# 3. Cargar en SQLite (Hive)
# 4. Consultar: top URLs, errores por hora


# === EJERCICIO 3: Bucketing simulado ===
# Tu código aquí:
# datos['bucket'] = datos['cliente_id'] % 4
# Demostrar JOIN eficiente leyendo solo 1 bucket


print("Completa los ejercicios arriba.")

## Recursos

- [Apache Pig UDFs](https://pig.apache.org/docs/latest/udf.html)
- [Hive Partitioned Tables](https://cwiki.apache.org/confluence/display/Hive/LanguageManual+DDL)
- [ORC File Format](https://orc.apache.org/)
- [Apache Parquet](https://parquet.apache.org/)

---

[← Sesión 27](./sesion-27.ipynb) | [Índice](../sesiones/index.md) | [Siguiente sesión →](./sesion-29.ipynb)